# Melbourne Housing Price Prediction - Decision Tree Model
This notebook implements a Decision Tree Regressor for Melbourne house prices

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

print("Libraries imported successfully!")

In [ ]:
# Load Melbourne Housing Data
melb_data = pd.read_csv("melb_data.csv")
print(f"Dataset shape: {melb_data.shape}")
melb_data.head()

In [ ]:
# Check for missing values in our selected features
features = ['Rooms', 'Bathroom', 'Landsize', 'Lattitude', 'Longtitude']

print("Missing values per column:")
print(melb_data[features].isnull().sum())

In [ ]:
# Handle missing values by dropping rows with missing values in our features
melbourne_model_data = melb_data.dropna(subset=features)
print(f"Filtered data shape: {melbourne_model_data.shape}")

In [ ]:
# Define target and features
y = melbourne_model_data.Price
X = melbourne_model_data[features]

print(f"Target variable (Price) shape: {y.shape}")
print(f"Features shape: {X.shape}")

In [ ]:
# Split data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, random_state=42)

(f"Training setprint size: {X_train.shape[0]}")
print(f"Validation set size: {X_val.shape[0]}")

In [ ]:
# Hyperparameter Tuning - Find optimal max_leaf_nodes
def get_mae(max_leaf_nodes, X_train, X_val, y_train, y_val):
    """Calculate MAE for different max_leaf_nodes values"""
    model = DecisionTreeRegressor(max_leaf_nodes=max_leaf_nodes, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    return mean_absolute_error(y_val, y_pred)

# Test different max_leaf_nodes values
max_leaf_nodes_options = [5, 10, 25, 50, 100, 150, 200, 250]
mae_results = []

for nodes in max_leaf_nodes_options:
    mae = get_mae(nodes, X_train, X_val, y_train, y_val)
    mae_results.append({'max_leaf_nodes': nodes, 'MAE': mae})
    print(f"max_leaf_nodes={nodes}: MAE = ${mae:,.2f}")

In [ ]:
# Find best max_leaf_nodes
results_df = pd.DataFrame(mae_results)
best_idx = results_df['MAE'].idxmin()
best_max_leaf_nodes = results_df.loc[best_idx, 'max_leaf_nodes']
best_mae = results_df.loc[best_idx, 'MAE']

print(f"\nBest max_leaf_nodes: {best_max_leaf_nodes}")
print(f"Best MAE: ${best_mae:,.2f}")

In [ ]:
# Train final model with optimal hyperparameters
final_model = DecisionTreeRegressor(
    max_leaf_nodes=best_max_leaf_nodes,
    random_state=42
)

final_model.fit(X_train, y_train)

print("Final Decision Tree model trained!")
print(final_model)

In [ ]:
# Final model evaluation
final_train_pred = final_model.predict(X_train)
final_val_pred = final_model.predict(X_val)

final_train_mae = mean_absolute_error(y_train, final_train_pred)
final_val_mae = mean_absolute_error(y_val, final_val_pred)

print("Final Model Performance:")
print(f"Training MAE: ${final_train_mae:,.2f}")
print(f"Validation MAE: ${final_val_mae:,.2f}")
print(f"\nThe model predicts Melbourne house prices with an average error of ~${final_val_mae/1000:,.0f}K")

In [ ]:
# Feature importance analysis
feature_importance = pd.DataFrame({
    'Feature': features,
    'Importance': final_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("Decision Tree Feature Importances:")
for idx, row in feature_importance.iterrows():
    print(f"{row['Feature']}: {row['Importance']:.3f}")

In [ ]:
# Summary
print("=" * 50)
print("SUMMARY - Decision Tree Model for Melbourne Housing")
print("=" * 50)
print(f"Model Type: {type(final_model).__name__}")
print(f"Optimal max_leaf_nodes: {best_max_leaf_nodes}")
print(f"Training samples: {X_train.shape[0]}")
print(f"Validation samples: {X_val.shape[0]}")
print(f"Training MAE: ${final_train_mae:,.2f}")
print(f"Validation MAE: ${final_val_mae:,.2f}")
print(f"Most important feature: {feature_importance.iloc[0]['Feature']} ({feature_importance.iloc[0]['Importance']:.3f})")
print(f"Second most important: {feature_importance.iloc[1]['Feature']} ({feature_importance.iloc[1]['Importance']:.3f})")